# Data Science In Action



# Imports

In [2]:
import pandas as pd
import numpy as np


## Data Load

In [19]:
cahoot_2015 = pd.read_csv("EPD_data/EugeneCAD2015noloc.csv", low_memory=False)
cahoot_2016 = pd.read_csv("EPD_data/EugeneCAD2016noloc.csv", low_memory=False)
cahoot_2017 = pd.read_csv("EPD_data/EugeneCAD2017noloc.csv", low_memory=False)
cahoot_2018 = pd.read_csv("EPD_data/EugeneCAD2018noloc.csv", low_memory=False)
cahoot_2019 = pd.read_csv("EPD_data/EugeneCAD2019noloc.csv", low_memory=False)
cahoot_2020 = pd.read_csv("EPD_data/EugeneCAD2020noloc.csv", low_memory=False)
cahoot_2021 = pd.read_csv("EPD_data/EugeneCAD2021noloc.csv", low_memory=False)
cahoot_2022 = pd.read_csv("EPD_data/EugeneCAD2022noloc.csv", low_memory=False)
cahoot_2023 = pd.read_csv("EPD_data/EugeneCAD2023noloc.csv", low_memory=False)
cahoot_2024 = pd.read_csv("EPD_data/EugeneCAD2024noloc.csv", low_memory=False)
cahoot_2025 = pd.read_csv("EPD_data/EugeneCAD2025noloc.csv", low_memory=False)

mcls = pd.read_excel("MCSLC.xlsx", sheet_name="Abbrev list")

print("CAHOOTS row counts by year (raw):")
for y, d in [(2015,cahoot_2015),(2016,cahoot_2016),(2017,cahoot_2017),(2018,cahoot_2018),
             (2019,cahoot_2019),(2020,cahoot_2020),(2021,cahoot_2021),(2022,cahoot_2022),
             (2023,cahoot_2023),(2024,cahoot_2024),(2025,cahoot_2025)]:
    print(f"  {y}: {len(d):>7,}  agencies: {sorted(d['agency'].astype(str).str.strip().unique())}")
print(f"\nMCLS rows: {len(mcls):,}")

CAHOOTS row counts by year (raw):
  2015: 123,248  agencies: ['EPD']
  2016: 127,732  agencies: ['EPD']
  2017: 129,688  agencies: ['EPD']
  2018: 127,432  agencies: ['EPD']
  2019: 137,089  agencies: ['EPD']
  2020: 137,287  agencies: ['EPD']
  2021: 137,527  agencies: ['EPD']
  2022: 140,545  agencies: ['CAHE', 'EPD']
  2023: 135,464  agencies: ['CAHE', 'EPD']
  2024: 132,650  agencies: ['CAHE', 'EPD']
  2025: 117,352  agencies: ['CAHE', 'EPD']

MCLS rows: 4,682


## Cahoot Clean

In [21]:
def clean_cahoots(df):
    df = df.copy()
    for col in ["agency", "nature", "closecode", "closed_as", "prime_unit"]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()
    df = df[df["agency"] == "CAHE"].reset_index(drop=True)
    return df

cahoot_2015 = clean_cahoots(cahoot_2015)
cahoot_2016 = clean_cahoots(cahoot_2016)
cahoot_2017 = clean_cahoots(cahoot_2017)
cahoot_2018 = clean_cahoots(cahoot_2018)
cahoot_2019 = clean_cahoots(cahoot_2019)
cahoot_2020 = clean_cahoots(cahoot_2020)
cahoot_2021 = clean_cahoots(cahoot_2021)
cahoot_2022 = clean_cahoots(cahoot_2022)
cahoot_2023 = clean_cahoots(cahoot_2023)
cahoot_2024 = clean_cahoots(cahoot_2024)
cahoot_2025 = clean_cahoots(cahoot_2025)

print("CAHOOTS row counts by year with filter:")
for y, d in [(2015, cahoot_2015),(2016,cahoot_2016),(2017,cahoot_2017),(2018,cahoot_2018),
             (2019,cahoot_2019),(2020,cahoot_2020),(2021,cahoot_2021),(2022,cahoot_2022),
             (2023,cahoot_2023),(2024,cahoot_2024),(2025,cahoot_2025)]:
    print(f"  cahoot_{y}: {len(d):>6,}")

CAHOOTS row counts by year with filter:
  cahoot_2015:      0
  cahoot_2016:      0
  cahoot_2017:      0
  cahoot_2018:      0
  cahoot_2019:      0
  cahoot_2020:      0
  cahoot_2021:      0
  cahoot_2022: 15,135
  cahoot_2023: 18,121
  cahoot_2024: 17,254
  cahoot_2025:  4,624


## MCLS Clean

In [22]:
mcls = mcls[["ID", "End Point of Dispatch", "Reason for Dispatch #1", "Disposition"]].copy()

mcls["End Point of Dispatch"]  = mcls["End Point of Dispatch"].astype("string").str.strip()
mcls["Reason for Dispatch #1"] = mcls["Reason for Dispatch #1"].astype("string").str.strip()
mcls["Disposition"] = mcls["Disposition"].astype("string").str.strip()


print(f"MCLS rows: {len(mcls):,}")
print("Columns:", list(mcls.columns))
print("\nDisposition value counts:")
print(mcls["Disposition"].value_counts(dropna=False))

MCLS rows: 4,682
Columns: ['ID', 'End Point of Dispatch', 'Reason for Dispatch #1', 'Disposition']

Disposition value counts:
Disposition
Remained in community         3750
Emergency Department           652
Other                          230
Sobering/Detox Facility         26
Arrest                          16
Crisis walk-in Center            3
Respite                          3
Sobering or Detox Facility       2
Name: count, dtype: Int64


## Build EPD_handoff - Cahoots

In [24]:
handoff_values = {
    "REFERRED TO OTHER AGENCY",
    "RELAYED TO LANE COUNTY SHERIFFS OFFICE",
    "RELAYED TO OREGON STATE POLICE",
    "UNIFORM TRAFFIC CITATION ISSUED",
}

def add_cahoots_handoff(df):
    df = df.copy()
    df["epd_handoff"] = df["closed_as"].isin(handoff_values).astype(int)
    return df

df_2022 = add_cahoots_handoff(cahoot_2022)
df_2023 = add_cahoots_handoff(cahoot_2023)
df_2024 = add_cahoots_handoff(cahoot_2024)
df_2025 = add_cahoots_handoff(cahoot_2025)

audit = pd.concat([df_2022, df_2023, df_2024, df_2025], ignore_index=True)
print("CAHOOTS closed_as × epd_handoff (2022–2025 combined):")
print(pd.crosstab(audit["closed_as"], audit["epd_handoff"], margins=True))

CAHOOTS closed_as × epd_handoff (2022–2025 combined):
epd_handoff                                 0    1    All
closed_as                                                
ACCIDENTALLY CHOSE NEW EVENT               71    0     71
ADVISED                                   294    0    294
ANIMAL WELFARE OFFICER                      1    0      1
ASSISTED                                25555    0  25555
CALLER CALLED BACK                        311    0    311
CALLER WILL CALL BACK                     204    0    204
CANCEL FIRE UNIT FROM CALL                  1    0      1
CANCEL WHILE ENROUTE                        1    0      1
DEAD ON ARRIVAL                             1    0      1
DISREGARD                                7552    0   7552
DISREGARDED BY DISPATCH                   743    0    743
DISREGARDED BY PATROL SUPERVISOR            4    0      4
EXECUTIVE ORDER 2012 VIOLATION              1    0      1
FALSE ALARM                                 3    0      3
FOLLOW UP INVESTIG

In [ ]:
mcls["epd_handoff"] = (mcls["Disposition"] == "Arrest").astype(int)

print("MCLS Disposition × epd_handoff:")
print(pd.crosstab(mcls["Disposition"], mcls["epd_handoff"], margins=True))
print(f"\nTotal handoffs: {mcls['epd_handoff'].sum():,} of {len(mcls):,} "
      f"({mcls['epd_handoff'].mean():.2%})")

MCLS Disposition × epd_handoff:
epd_handoff                    0   1   All
Disposition                               
Arrest                         0  16    16
Crisis walk-in Center          3   0     3
Emergency Department         652   0   652
Other                        230   0   230
Remained in community       3750   0  3750
Respite                        3   0     3
Sobering or Detox Facility     2   0     2
Sobering/Detox Facility       26   0    26
All                         4666  16  4682

Total handoffs: 16 of 4,682 (0.341734%)


In [ ]:
mcls = mcls[["ID", "End Point of Dispatch", "Reason for Dispatch #1", "Disposition"]].copy()

mcls["End Point of Dispatch"] = mcls["End Point of Dispatch"].astype("string").str.strip()
mcls["Reason for Dispatch #1"] = mcls["Reason for Dispatch #1"].astype("string").str.strip()
mcls["Disposition"] = mcls["Disposition"].astype("string").str.strip()

print(f"MCLS rows: {len(mcls):,}")
print("Columns:", list(mcls.columns))
print("\nDisposition value counts:")
print(mcls["Disposition"].value_counts(dropna=False))

MCLS rows: 4,682
Columns: ['ID', 'End Point of Dispatch', 'Reason for Dispatch #1', 'Disposition']

Disposition value counts:
Disposition
Remained in community         3750
Emergency Department           652
Other                          230
Sobering/Detox Facility         26
Arrest                          16
Crisis walk-in Center            3
Respite                          3
Sobering or Detox Facility       2
Name: count, dtype: Int64
